# Beer Peer
## Advanced Machine Learning — Final Project

---

### Product Description

Beer Peer is a beer recommendation app. Photograph your food — that is the only input. The app returns three beer style recommendations with flavor descriptions written by real beer drinkers.

The pipeline is three independent layers:

```
PHOTO  →  CNN  →  food label  →  lookup table  →  beer styles  →  BiLSTM  →  flavor language
```

---

### Example Output

| | |
|---|---|
| **Input** | Photo of pizza margherita |
| **CNN output** | Pizza (94% confidence) |
| **Paired styles** | IPA · APA · Lager |
| **IPA** | "Citrusy, hoppy, bitter finish that cuts through rich tomato sauce and melted cheese." |
| **APA** | "Balanced hop character with a clean malt backbone — lifts the saltiness without overpowering." |
| **Lager** | "Crisp and clean. Light bitterness that refreshes the palate between bites." |

---

### Models

| Component | Dataset | Task |
|---|---|---|
| CNN (scratch) | Food-101 (101k images) | 101-class food classification |
| CNN (ResNet-50) | Food-101 | Same — compare vs. scratch |
| LSTM baseline | BeerAdvocate (1.5M reviews) | 8-class beer macro-style classification |
| BiLSTM + attention | BeerAdvocate | Same — compare vs. unidirectional |
| Joint Model *(+10 pts)* | Food-101 + BeerAdvocate pairs | Food-beer compatibility: 0/1 |

---

### Joint Model Design

The joint model learns food-beer compatibility from pairs supervised by the lookup table:

- **Positive pairs:** (food image, beer review) where the lookup table says they pair → label `1`
- **Negative pairs:** (food image, beer review) for a non-paired style → label `0`
- **Architecture:** frozen CNN encoder (512-d) + frozen BiLSTM encoder (256-d) → concat (768-d) → FC → sigmoid

After training, the joint model can score food-beer pairs **not in the lookup table** — generalizing the hardcoded rules into a learned compatibility function.

---

### Datasets

| Dataset | Content | Used for |
|---|---|---|
| `ethz/food101` | 101,000 labeled food images | CNN training |
| BeerAdvocate (`rdoume/beerreviews`) | 1.5M reviews, beer_style, review_text | BiLSTM training |
| `food_beer_pairing.csv` (embedded) | 101 foods × 3 beer styles | Lookup + joint model labels |

**BiLSTM labels:** 100+ raw beer styles → 8 macro-classes (Lager / IPA / Stout-Porter / Wheat / Sour-Farmhouse / Amber-Brown / Pale Ale / Specialty)

---

### Development Phases

| Phase | Environment | Sections | Needs GPU | What happens |
|---|---|---|---|---|
| **Phase 1** | VS Code (CPU) | 1, 2, 3, 4, 6, 10, 11 | No | Environment setup. Data loading. EDA (images + text). Text preprocessing. LSTM baseline. BiLSTM with attention. |
| **Phase 2** | Google Colab (GPU) | 5, 7, 8, 13 | **Yes** | Image preprocessing + data loaders. CNN from scratch. CNN ResNet-50. Joint model full training. |
| **Phase 3** | VS Code (CPU) | 9, 12, 14, 15, deployment | No | Grad-CAM explainability. BiLSTM attention explainability. Recommendation card + 20-example table. Business framing + ethics. HF Spaces deployment. |

---

### Notebook Structure

| Section | Content |
|---|---|
| 1 | Environment setup — dependencies and imports |
| 2 | Data loading — Food-101 + BeerAdvocate + pairing table |
| 3 | EDA — image dataset (class distribution, sample grid) |
| 4 | EDA — text dataset (review length, style distribution, word clouds) |
| 5 | Image preprocessing and data loaders |
| 6 | Text preprocessing and data loaders |
| 7 | CNN — custom architecture (trained from scratch) |
| 8 | CNN — ResNet-50 (transfer learning) |
| 9 | CNN explainability — Grad-CAM |
| 10 | LSTM — unidirectional baseline |
| 11 | BiLSTM — bidirectional with attention |
| 12 | BiLSTM explainability — attention weights |
| 13 | Joint model — food-beer compatibility classifier *(+10 bonus)* |
| 14 | Business integration — recommendation card and 20-example table |
| 15 | Business framing, ethics, and team contributions |

---


## Section 1 — Environment Setup

### 1.1 — pip installs

If you need to install all required packages, run the cell below.
It auto-detects the environment (local CPU vs. Google Colab GPU) and installs only what is missing.


In [3]:
import subprocess, sys

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

# ── Detect environment ────────────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

# ── Install PyTorch with correct backend ──────────────────────────────────────
try:
    import torch
    print(f"  OK  torch {torch.__version__}")
except ImportError:
    print("  INSTALLING  torch ...")
    if IN_COLAB:
        pip_install("torch", "torchvision", "torchaudio",
                    "--index-url", "https://download.pytorch.org/whl/cu118")
    else:
        pip_install("torch", "torchvision", "torchaudio",
                    "--index-url", "https://download.pytorch.org/whl/cpu")
    print("  DONE  torch")

# ── Install remaining packages ────────────────────────────────────────────────
PACKAGES = {
    "datasets":   "datasets",
    "kaggle":     "kaggle",
    "nltk":       "nltk",
    "torchcam":   "torchcam",
    "wordcloud":  "wordcloud",
    "matplotlib": "matplotlib",
    "seaborn":    "seaborn",
    "pandas":     "pandas",
    "sklearn":    "scikit-learn",
    "PIL":        "Pillow",
}

for pkg, install_name in PACKAGES.items():
    try:
        __import__(pkg)
        print(f"  OK  {pkg}")
    except ImportError:
        print(f"  INSTALLING  {pkg} ...")
        pip_install(install_name)
        print(f"  DONE  {pkg}")

print("\nAll packages ready.")


Environment: Local
  OK  torch 2.11.0+cpu
  OK  datasets
  INSTALLING  kaggle ...
  DONE  kaggle
  OK  nltk
  OK  torchcam
  OK  wordcloud
  OK  matplotlib
  OK  seaborn
  OK  pandas
  OK  sklearn
  OK  PIL

All packages ready.


### 1.2 — Library imports

Import all libraries, set the random seed, detect the device, and create project folders.


In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import random
import string
import importlib
from pathlib import Path
from collections import Counter

# ── Numeric / data ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from wordcloud import WordCloud
from PIL import Image

# ── PyTorch core ──────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ── Computer vision ───────────────────────────────────────────────────────────
import torchvision.transforms as T
import torchvision.models as models

# ── NLP ───────────────────────────────────────────────────────────────────────
import nltk
from nltk.tokenize import word_tokenize

# ── HuggingFace datasets (Food-101) ───────────────────────────────────────────
from datasets import load_dataset

# ── Sklearn utilities ─────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Project directories ───────────────────────────────────────────────────────
BASE_DIR = Path(".")
WEIGHTS  = BASE_DIR / "weights"
FIGURES  = BASE_DIR / "figures"
DEPLOY   = BASE_DIR / "deployment"

for d in [WEIGHTS, FIGURES, DEPLOY]:
    d.mkdir(exist_ok=True)

# ── NLTK data ─────────────────────────────────────────────────────────────────
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

# ── Library versions ──────────────────────────────────────────────────────────
libs = [
    "torch", "torchvision", "numpy", "pandas",
    "matplotlib", "seaborn", "PIL", "sklearn",
    "nltk", "datasets", "wordcloud",
]

print(f"{'Library':<20} {'Version'}")
print("-" * 35)
for lib in libs:
    try:
        mod = importlib.import_module(lib)
        print(f"{lib:<20} {getattr(mod, '__version__', 'n/a')}")
    except ImportError:
        print(f"{lib:<20} NOT INSTALLED")

print()
print(f"{'Device':<20} {DEVICE}")
print(f"{'CUDA available':<20} {torch.cuda.is_available()}")
print(f"{'Directories':<20} {[str(d) for d in [WEIGHTS, FIGURES, DEPLOY]]}")


Library              Version
-----------------------------------
torch                2.11.0+cpu
torchvision          0.26.0+cpu
numpy                2.4.4
pandas               3.0.2
matplotlib           3.10.8
seaborn              0.13.2
PIL                  12.1.1
sklearn              1.8.0
nltk                 3.9.4
datasets             2.21.0
wordcloud            1.9.6

Device               cpu
CUDA available       False
Directories          ['weights', 'figures', 'deployment']

Section 1 complete.
